In [14]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import(
    accuracy_score,
    classification_report,
    confusion_matrix 
)
import pickle
import warnings
warnings.filterwarnings("ignore")

In [15]:
reimbursement = pd.read_excel(
    "../datasets/improved_reimbursement_fraud_dataset.xlsx"
)

reimbursement.head()

,Employee_ID,Department,Expense_Type,Claim_Amount,Claim_Date,Approval_Status,Fraud_Label,Designation,Employee_Tenure_Months,Manager_ID,Receipt_Available,Duplicate_Claim,Previous_Claims_Count,Approval_Time_Days,Weekend_Claim,Policy_Violation_Count,Payment_Method,Risk_Score
0,E621,HR,Fuel,10973.03,2024-06-20,Approved,0,Associate,93,M015,Yes,No,4,8,No,0,Bank Transfer,40
1,E354,Operations,Hotel,2036.92,2024-07-23,Pending,0,Senior Associate,61,M021,Yes,No,9,1,No,0,Bank Transfer,7
2,E823,IT,Travel,3813.36,2025-01-21,Pending,0,Associate,83,M087,Yes,No,14,4,No,1,Corporate Card,18
3,E710,Sales,Training,14152.43,2024-12-20,Approved,0,Intern,88,M100,Yes,No,6,6,No,0,Bank Transfer,19
4,E123,Finance,Hotel,1920.93,2025-08-12,Pending,0,Associate,3,M022,Yes,No,6,8,No,1,Bank Transfer,18


In [16]:
print(reimbursement.shape)

reimbursement.info()

(5000, 18)
<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 18 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   Employee_ID             5000 non-null   str           
 1   Department              5000 non-null   str           
 2   Expense_Type            5000 non-null   str           
 3   Claim_Amount            5000 non-null   float64       
 4   Claim_Date              5000 non-null   datetime64[us]
 5   Approval_Status         5000 non-null   str           
 6   Fraud_Label             5000 non-null   int64         
 7   Designation             5000 non-null   str           
 8   Employee_Tenure_Months  5000 non-null   int64         
 9   Manager_ID              5000 non-null   str           
 10  Receipt_Available       5000 non-null   str           
 11  Duplicate_Claim         5000 non-null   str           
 12  Previous_Claims_Count   5000 non-null   int64   

In [17]:
reimbursement["Fraud_Label"].value_counts()

Fraud_Label
0    4594
1     406
Name: count, dtype: int64

In [18]:
df = reimbursement.copy()

df.head()


,Employee_ID,Department,Expense_Type,Claim_Amount,Claim_Date,Approval_Status,Fraud_Label,Designation,Employee_Tenure_Months,Manager_ID,Receipt_Available,Duplicate_Claim,Previous_Claims_Count,Approval_Time_Days,Weekend_Claim,Policy_Violation_Count,Payment_Method,Risk_Score
0,E621,HR,Fuel,10973.03,2024-06-20,Approved,0,Associate,93,M015,Yes,No,4,8,No,0,Bank Transfer,40
1,E354,Operations,Hotel,2036.92,2024-07-23,Pending,0,Senior Associate,61,M021,Yes,No,9,1,No,0,Bank Transfer,7
2,E823,IT,Travel,3813.36,2025-01-21,Pending,0,Associate,83,M087,Yes,No,14,4,No,1,Corporate Card,18
3,E710,Sales,Training,14152.43,2024-12-20,Approved,0,Intern,88,M100,Yes,No,6,6,No,0,Bank Transfer,19
4,E123,Finance,Hotel,1920.93,2025-08-12,Pending,0,Associate,3,M022,Yes,No,6,8,No,1,Bank Transfer,18


In [19]:
reimbursement.isnull().sum()

Employee_ID               0
Department                0
Expense_Type              0
Claim_Amount              0
Claim_Date                0
Approval_Status           0
Fraud_Label               0
Designation               0
Employee_Tenure_Months    0
Manager_ID                0
Receipt_Available         0
Duplicate_Claim           0
Previous_Claims_Count     0
Approval_Time_Days        0
Weekend_Claim             0
Policy_Violation_Count    0
Payment_Method            0
Risk_Score                0
dtype: int64

In [20]:
df = df.drop(
    [
        "Employee_ID",
        "Claim_Date",
        "Manager_ID"
    ],
    axis=1
)

df.head()


,Department,Expense_Type,Claim_Amount,Approval_Status,Fraud_Label,Designation,Employee_Tenure_Months,Receipt_Available,Duplicate_Claim,Previous_Claims_Count,Approval_Time_Days,Weekend_Claim,Policy_Violation_Count,Payment_Method,Risk_Score
0,HR,Fuel,10973.03,Approved,0,Associate,93,Yes,No,4,8,No,0,Bank Transfer,40
1,Operations,Hotel,2036.92,Pending,0,Senior Associate,61,Yes,No,9,1,No,0,Bank Transfer,7
2,IT,Travel,3813.36,Pending,0,Associate,83,Yes,No,14,4,No,1,Corporate Card,18
3,Sales,Training,14152.43,Approved,0,Intern,88,Yes,No,6,6,No,0,Bank Transfer,19
4,Finance,Hotel,1920.93,Pending,0,Associate,3,Yes,No,6,8,No,1,Bank Transfer,18


In [21]:
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()

for column in df.columns:
    if df[column].dtype == "object":
        df[column] = encoder.fit_transform(df[column])

df.head()

,Department,Expense_Type,Claim_Amount,Approval_Status,Fraud_Label,Designation,Employee_Tenure_Months,Receipt_Available,Duplicate_Claim,Previous_Claims_Count,Approval_Time_Days,Weekend_Claim,Policy_Violation_Count,Payment_Method,Risk_Score
0,HR,Fuel,10973.03,Approved,0,Associate,93,Yes,No,4,8,No,0,Bank Transfer,40
1,Operations,Hotel,2036.92,Pending,0,Senior Associate,61,Yes,No,9,1,No,0,Bank Transfer,7
2,IT,Travel,3813.36,Pending,0,Associate,83,Yes,No,14,4,No,1,Corporate Card,18
3,Sales,Training,14152.43,Approved,0,Intern,88,Yes,No,6,6,No,0,Bank Transfer,19
4,Finance,Hotel,1920.93,Pending,0,Associate,3,Yes,No,6,8,No,1,Bank Transfer,18


In [22]:
X = df.drop("Fraud_Label", axis=1)

y = df["Fraud_Label"]


In [23]:
print(X.shape)
print(y.shape)

(5000, 14)
(5000,)


In [24]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [25]:
print(X_train.shape)
print(X_test.shape)

(4000, 14)
(1000, 14)


In [26]:
from sklearn.ensemble import RandomForestClassifier

reimbursement_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

reimbursement_model.fit(
    X_train,
    y_train
)

ValueError: could not convert string to float: 'IT'

In [27]:
df.select_dtypes(include="object").columns

Index(['Department', 'Expense_Type', 'Approval_Status', 'Designation',
       'Receipt_Available', 'Duplicate_Claim', 'Weekend_Claim',
       'Payment_Method'],
      dtype='str')

In [28]:
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()

categorical_cols = [
    'Department',
    'Expense_Type',
    'Approval_Status',
    'Designation',
    'Receipt_Available',
    'Duplicate_Claim',
    'Payment_Method'
]

for col in categorical_cols:
    df[col] = encoder.fit_transform(df[col])

df.head()

,Department,Expense_Type,Claim_Amount,Approval_Status,Fraud_Label,Designation,Employee_Tenure_Months,Receipt_Available,Duplicate_Claim,Previous_Claims_Count,Approval_Time_Days,Weekend_Claim,Policy_Violation_Count,Payment_Method,Risk_Score
0,1,0,10973.03,0,0,0,93,1,0,4,8,No,0,0,40
1,3,1,2036.92,1,0,4,61,1,0,9,1,No,0,0,7
2,2,6,3813.36,1,0,0,83,1,0,14,4,No,1,2,18
3,4,5,14152.43,0,0,2,88,1,0,6,6,No,0,0,19
4,0,1,1920.93,1,0,0,3,1,0,6,8,No,1,0,18


In [29]:
df.dtypes

Department                  int64
Expense_Type                int64
Claim_Amount              float64
Approval_Status             int64
Fraud_Label                 int64
Designation                 int64
Employee_Tenure_Months      int64
Receipt_Available           int64
Duplicate_Claim             int64
Previous_Claims_Count       int64
Approval_Time_Days          int64
Weekend_Claim                 str
Policy_Violation_Count      int64
Payment_Method              int64
Risk_Score                  int64
dtype: object

In [30]:
X = df.drop("Fraud_Label", axis=1)
y = df["Fraud_Label"]

In [31]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

Training data: (4000, 14)
Testing data: (1000, 14)


In [32]:
from sklearn.ensemble import RandomForestClassifier

reimbursement_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    class_weight="balanced"
)

reimbursement_model.fit(
    X_train,
    y_train
)

ValueError: could not convert string to float: 'No'

In [33]:
df.select_dtypes(include=['object']).columns

Index(['Weekend_Claim'], dtype='str')

In [34]:
from sklearn.preprocessing import LabelEncoder

for col in df.columns:
    if df[col].dtype == 'object':
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col])

df.head()

,Department,Expense_Type,Claim_Amount,Approval_Status,Fraud_Label,Designation,Employee_Tenure_Months,Receipt_Available,Duplicate_Claim,Previous_Claims_Count,Approval_Time_Days,Weekend_Claim,Policy_Violation_Count,Payment_Method,Risk_Score
0,1,0,10973.03,0,0,0,93,1,0,4,8,No,0,0,40
1,3,1,2036.92,1,0,4,61,1,0,9,1,No,0,0,7
2,2,6,3813.36,1,0,0,83,1,0,14,4,No,1,2,18
3,4,5,14152.43,0,0,2,88,1,0,6,6,No,0,0,19
4,0,1,1920.93,1,0,0,3,1,0,6,8,No,1,0,18


In [35]:
df.dtypes

Department                  int64
Expense_Type                int64
Claim_Amount              float64
Approval_Status             int64
Fraud_Label                 int64
Designation                 int64
Employee_Tenure_Months      int64
Receipt_Available           int64
Duplicate_Claim             int64
Previous_Claims_Count       int64
Approval_Time_Days          int64
Weekend_Claim                 str
Policy_Violation_Count      int64
Payment_Method              int64
Risk_Score                  int64
dtype: object

In [36]:
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()

df["Weekend_Claim"] = encoder.fit_transform(
    df["Weekend_Claim"]
)

In [37]:
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()

df["Weekend_Claim"] = encoder.fit_transform(
    df["Weekend_Claim"]
)

In [38]:
df.dtypes

Department                  int64
Expense_Type                int64
Claim_Amount              float64
Approval_Status             int64
Fraud_Label                 int64
Designation                 int64
Employee_Tenure_Months      int64
Receipt_Available           int64
Duplicate_Claim             int64
Previous_Claims_Count       int64
Approval_Time_Days          int64
Weekend_Claim               int64
Policy_Violation_Count      int64
Payment_Method              int64
Risk_Score                  int64
dtype: object

In [39]:
X = df.drop("Fraud_Label", axis=1)

y = df["Fraud_Label"]

print(X.shape)
print(y.shape)

(5000, 14)
(5000,)


In [40]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(X_train.shape)
print(X_test.shape)

(4000, 14)
(1000, 14)


In [41]:
from sklearn.ensemble import RandomForestClassifier

reimbursement_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    class_weight="balanced"
)

reimbursement_model.fit(
    X_train,
    y_train
)

,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"class_weight class_weight: {""balanced"", ""balanced_subsample""}, dict or list of dicts, default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one. Formulti-output problems, a list of dicts can be provided in the sameorder as the columns of y.Note that for multioutput (including multilabel) weights should bedefined for each class of every column in its own dict. For example,for four-class multilabel classification weights should be[{0: 1, 1: 1}, {0: 1, 1: 5}, {0: 1, 1: 1}, {0: 1, 1: 1}] instead of[{1:1}, {2:5}, {3:1}, {4:1}].The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``The ""balanced_subsample"" mode is the same as ""balanced"" except thatweights are computed based on the bootstrap sample for every treegrown.For multi-output, the weights of each column of y will be multiplied.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified.",'balanced'
,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.

In [42]:
y_pred = reimbursement_model.predict(X_test)

y_pred[:20]

array([0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0])

In [43]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("Accuracy:")
print(
    accuracy_score(
        y_test,
        y_pred
    )
)

print("\nConfusion Matrix:")
print(
    confusion_matrix(
        y_test,
        y_pred
    )
)

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred
    )
)

Accuracy:
1.0

Confusion Matrix:
[[919   0]
 [  0  81]]

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       919
           1       1.00      1.00      1.00        81

    accuracy                           1.00      1000
   macro avg       1.00      1.00      1.00      1000
weighted avg       1.00      1.00      1.00      1000



In [44]:
X = df.drop(
    [
        "Fraud_Label",
        "Risk_Score"
    ],
    axis=1
)

y = df["Fraud_Label"]

In [45]:
X = df.drop(
    [
        "Fraud_Label",
        "Risk_Score"
    ],
    axis=1
)

y = df["Fraud_Label"]

print(X.shape)
print(y.shape)

(5000, 13)
(5000,)


In [46]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [47]:
final_reimbursement_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    class_weight="balanced"
)

final_reimbursement_model.fit(
    X_train,
    y_train
)

,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"class_weight class_weight: {""balanced"", ""balanced_subsample""}, dict or list of dicts, default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one. Formulti-output problems, a list of dicts can be provided in the sameorder as the columns of y.Note that for multioutput (including multilabel) weights should bedefined for each class of every column in its own dict. For example,for four-class multilabel classification weights should be[{0: 1, 1: 1}, {0: 1, 1: 5}, {0: 1, 1: 1}, {0: 1, 1: 1}] instead of[{1:1}, {2:5}, {3:1}, {4:1}].The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``The ""balanced_subsample"" mode is the same as ""balanced"" except thatweights are computed based on the bootstrap sample for every treegrown.For multi-output, the weights of each column of y will be multiplied.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified.",'balanced'
,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.

In [48]:
y_pred = final_reimbursement_model.predict(
    X_test
)

In [49]:
print("Accuracy:")
print(
    accuracy_score(
        y_test,
        y_pred
    )
)

print("\nConfusion Matrix:")
print(
    confusion_matrix(
        y_test,
        y_pred
    )
)

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred
    )
)

Accuracy:
1.0

Confusion Matrix:
[[919   0]
 [  0  81]]

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       919
           1       1.00      1.00      1.00        81

    accuracy                           1.00      1000
   macro avg       1.00      1.00      1.00      1000
weighted avg       1.00      1.00      1.00      1000



In [50]:
import pickle

with open(
    "../models/reimbursement_model.pkl",
    "wb"
) as file:
    pickle.dump(
        final_reimbursement_model,
        file
    )

In [51]:
import os

os.listdir("../models")

['reimbursement_model.pkl']

In [52]:
feature_columns = list(X.columns)

with open(
    "../models/reimbursement_features.pkl",
    "wb"
) as file:
    pickle.dump(
        feature_columns,
        file
    )

In [53]:
with open("../models/reimbursement_model.pkl", "rb") as file:
    loaded_model = pickle.load(file)


loaded_model.predict(X_test[:5])

array([0, 0, 0, 1, 0])